In [1]:
import pandas as pd
import re

In [2]:
# Define columns needed and create dataframes from IPEDS 2011 and 2021

cols = ["UNITID", "CIPCODE", "AWLEVEL", "CTOTALT", "CTOTALM", "CTOTALW"]
c2011_clean = pd.read_csv(
    r"C:\capstone_data\raw\c2011_a.csv", 
    usecols=cols,
    dtype={"CIPCODE":str}
)

In [3]:
c2021_clean = pd.read_csv(
    r"C:\capstone_data\raw\c2021_a.csv", 
    usecols=cols,
    dtype={"CIPCODE":str}
)

In [4]:
# Dictionary for renaming columns

rename_dict = {
    "UNITID" : "unitid"
    ,"CIPCODE" : "cipcode"
    ,"AWLEVEL" : "credlev"
    ,"CTOTALT" : "completions"
    ,"CTOTALM" : "completions_men"
    ,"CTOTALW" : "completions_women"
}

In [5]:
# Rename the columns

c2011_clean = c2011_clean.rename(columns=rename_dict)
c2021_clean = c2021_clean.rename(columns=rename_dict)

In [6]:
print(c2011_clean.columns == c2021_clean.columns)

[ True  True  True  True  True  True]


In [7]:
# Set years

c2011_clean["year"] = 2011
c2021_clean["year"] = 2021

In [8]:
# "Stack" the two sets of data so that 2011 sits on top of 2021

completions = pd.concat([c2011_clean, c2021_clean], ignore_index=True)
completions.shape

(562151, 7)

In [9]:
c2011_clean["credlev"].value_counts().sort_index()
c2021_clean["credlev"].value_counts().sort_index()

credlev
2      30085
3      48506
4       2578
5     103820
6      12450
7      43913
8       5179
17     13130
18      2866
19       383
20      3762
21     29671
Name: count, dtype: int64

In [10]:
# Create a dictionary for the credlev codes, limiting it to the ones we are interested in

level_map = {
    2 : "Short Certificate"
    ,3 : "Long Certificate"
    ,5 : "Long Certificate"
    ,4 : "Associate"
    ,6 : "Bachelor"
}
completions["credential_group"] = completions["credlev"].map(level_map)

In [11]:
# Format ipeds cip code to xxxx dtype str format. Maintain cip code 99 for accuracy.

def ipeds_to_4digit(cip):
    s = str(cip)
    digits = "".join(re.findall(r"\d+", s))    # Extract the digits
    if digits == "99":
        return "9900"                         # Special: cip 99 to 9900
    digits = digits.zfill(6)
    return digits [:4]

completions["cip4"] = completions["cipcode"].apply(ipeds_to_4digit)

In [12]:
# Checkpoint to see if everything is looking good.

completions["year"].value_counts()

year
2021    296343
2011    265808
Name: count, dtype: int64

In [13]:
completions["credlev"].value_counts()

credlev
5     207121
3      96997
7      79758
2      59204
21     29671
1      25020
17     23747
6      17565
8       8312
4       5521
18      4794
20      3762
19       679
Name: count, dtype: int64

In [14]:
completions.head(8)

,unitid,cipcode,credlev,completions,completions_men,completions_women,year,credential_group,cip4
0,100636,09.0999,3,66,34,32,2011,Long Certificate,0909
1,100636,10.0105,3,547,398,149,2011,Long Certificate,1001
2,100636,11.0101,3,69,65,4,2011,Long Certificate,1101
3,100636,11.0401,3,915,705,210,2011,Long Certificate,1104
4,100636,13.0499,3,269,144,125,2011,Long Certificate,1304
5,100636,13.9999,3,1486,1244,242,2011,Long Certificate,1399
6,100636,13.9999,4,993,812,181,2011,Associate,1399
7,100636,15.0303,3,1012,913,99,2011,Long Certificate,1503


In [15]:
completions.tail(8)

,unitid,cipcode,credlev,completions,completions_men,completions_women,year,credential_group,cip4
562143,497329,51.0801,2,0,0,0,2021,Short Certificate,5108
562144,497329,51.3501,21,0,0,0,2021,NaN,5135
562145,497329,99,2,0,0,0,2021,Short Certificate,9900
562146,497329,99,21,0,0,0,2021,NaN,9900
562147,497338,51.0801,21,0,0,0,2021,NaN,5108
562148,497338,51.3801,3,0,0,0,2021,Long Certificate,5138
562149,497338,99,3,0,0,0,2021,Long Certificate,9900
562150,497338,99,21,0,0,0,2021,NaN,9900


In [16]:
completions.shape

(562151, 9)

In [17]:
# Load College Scorecard Field of Study data

scorecard_fos = pd.read_csv(r"C:\capstone_data\raw\Most-Recent-Cohorts-Field-of-Study.csv")

In [18]:
scorecard_fos.shape

(229188, 174)

In [19]:
scorecard_fos.columns

Index(['UNITID', 'OPEID6', 'INSTNM', 'CONTROL', 'MAIN', 'CIPCODE', 'CIPDESC',
       'CREDLEV', 'CREDDESC', 'IPEDSCOUNT1',
       ...
       'EARN_COUNT_PELL_WNE_5YR', 'EARN_PELL_WNE_MDN_5YR',
       'EARN_COUNT_NOPELL_WNE_5YR', 'EARN_NOPELL_WNE_MDN_5YR',
       'EARN_COUNT_MALE_WNE_5YR', 'EARN_MALE_WNE_MDN_5YR',
       'EARN_COUNT_NOMALE_WNE_5YR', 'EARN_NOMALE_WNE_MDN_5YR',
       'EARN_COUNT_HIGH_CRED_5YR', 'EARN_IN_STATE_5YR'],
      dtype='object', length=174)

In [20]:
# Find earnings columns that may be useful later

[col for col in scorecard_fos.columns if "MDN_3YR".lower() in col.lower()]

['EARN_NE_MDN_3YR',
 'EARN_PELL_NE_MDN_3YR',
 'EARN_NOPELL_NE_MDN_3YR',
 'EARN_MALE_NE_MDN_3YR',
 'EARN_NOMALE_NE_MDN_3YR']

In [21]:
# Select columns from scorecard to keep. EARN_NE_MDN_3YR provides median earnings 3 years after completion for non-enrolled students.

scorecard_fos_clean = scorecard_fos[[
    "UNITID"
    ,"INSTNM"
    ,"CIPCODE"
    ,"EARN_MDN_1YR"
    ,"EARN_NE_MDN_3YR"
    ,"EARN_MDN_5YR"
]].copy()

In [22]:
scorecard_fos_clean.rename(columns={
    "UNITID" : "unitid"
    ,"INSTNM" : "inst_name"
    ,"CIPCODE" : "cipcode"
    ,"EARN_MDN_1YR" : "median_earnings_1yr"
    ,"EARN_NE_MDN_3YR" : "median_earnings_3yr"
    ,"EARN_MDN_5YR" : "median_earnings_5yr"
}, inplace = True)

In [23]:
# Fix cip code to uniform xxxx dtype str format.
# Fill three digit codes to 4 digit with leading 0

scorecard_fos_clean["cip4"]= (
    scorecard_fos_clean["cipcode"]
    .astype(str)
    .str.zfill(4)
)

In [24]:
# Load College Scorecard Institution Level data

scorecard_il = pd.read_csv(r"C:\capstone_data\raw\Most-Recent-Cohorts-Institution.csv")

C:\Users\sigep\AppData\Local\Temp\ipykernel_60996\3901106909.py:3: DtypeWarning: Columns (9,1407,1408,1431,1432,1532,1537,1538,1539,1540,1542,1546,1589,1601,1602,1606,1608,1611,1614,1615,1616,1619,1620,1621,1622,1623,1624,1625,1626,1627,1628,1629,1653,1679,1690,1692,1697,1700,1702,1725,1726,1727,1728,1729,1743,1815,1816,1817,1818,1823,1824,1830,1831,1879,1880,1881,1882,1883,1884,1885,1886,1887,1888,1889,1890,1891,1892,1893,1894,1895,1896,1897,1898,1909,1910,1911,1912,1913,1957,1958,1959,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1983,1984,2376,2377,2403,2404,2495,2496,2497,2498,2499,2500,2501,2502,2503,2504,2505,2506,2507,2508,2509,2510,2511,2512,2513,2514,2515,2516,2517,2518,2519,2520,2521,2522,2523,2524,2525,2526,2527,2528,2529,2530,2958,3215,3231,3235,3236) have mixed types. Specify dtype option on import or set low_memory=False.
  scorecard_il = pd.read_csv(r"C:\capstone_data\raw\Most-Recent-Cohorts-Institution.csv")


In [25]:
scorecard_il.shape

(6429, 3306)

In [26]:
# Select columns from scorecard to keep.

scorecard_il_clean = scorecard_il[[
    "UNITID"
    ,"STABBR"
    ,"TUITIONFEE_IN"
    ,"TUITIONFEE_OUT"
    ,"NPT4_PUB"
    ,"NPT4_PRIV"
    ,"COSTT4_A"
]].copy()

In [27]:
# Clean columns for naming conventions

scorecard_il_clean.rename(columns={
    "UNITID" : "unitid"
    ,"STABBR" : "state"
    ,"TUITIONFEE_IN" : "tuition_in"
    ,"TUITIONFEE_OUT" : "tuition_out"
    ,"NPT4_PUB" : "avg_net_price_pub"
    ,"NPT4_PRIV" : "avg_net_price_priv"
    ,"COSTT4_A" : "avg_annual_cost"
}, inplace = True)

In [28]:
scorecard_il_clean.columns

Index(['unitid', 'state', 'tuition_in', 'tuition_out', 'avg_net_price_pub',
       'avg_net_price_priv', 'avg_annual_cost'],
      dtype='object')

In [29]:
# Get unitids for Tennessee institutions

tn_unitids = scorecard_il_clean[scorecard_il_clean["state"] == "TN"]["unitid"]

# Filter the field of study dataframe to only include Tennessee institutions

scorecard_fos_tn = scorecard_fos_clean[scorecard_fos_clean["unitid"].isin(tn_unitids)]

# Filter the institution-level dataframe to only include Tennessee institutions
scorecard_il_tn = scorecard_il_clean[scorecard_il_clean["state"] == "TN"]

In [30]:
scorecard_il_tn.info()

<class 'pandas.core.frame.DataFrame'>
Index: 153 entries, 2961 to 6379
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   unitid              153 non-null    int64  
 1   state               153 non-null    object 
 2   tuition_in          71 non-null     float64
 3   tuition_out         71 non-null     float64
 4   avg_net_price_pub   46 non-null     float64
 5   avg_net_price_priv  82 non-null     float64
 6   avg_annual_cost     66 non-null     float64
dtypes: float64(5), int64(1), object(1)
memory usage: 9.6+ KB


In [31]:
scorecard_fos_tn.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4166 entries, 162267 to 219051
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   unitid               4166 non-null   float64
 1   inst_name            4166 non-null   object 
 2   cipcode              4166 non-null   int64  
 3   median_earnings_1yr  4166 non-null   object 
 4   median_earnings_3yr  3681 non-null   object 
 5   median_earnings_5yr  4166 non-null   object 
 6   cip4                 4166 non-null   object 
dtypes: float64(1), int64(1), object(5)
memory usage: 260.4+ KB


In [32]:
completions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 562151 entries, 0 to 562150
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   unitid             562151 non-null  int64 
 1   cipcode            562151 non-null  object
 2   credlev            562151 non-null  int64 
 3   completions        562151 non-null  int64 
 4   completions_men    562151 non-null  int64 
 5   completions_women  562151 non-null  int64 
 6   year               562151 non-null  int64 
 7   credential_group   386408 non-null  object
 8   cip4               562151 non-null  object
dtypes: int64(6), object(3)
memory usage: 38.6+ MB


In [33]:
# Is NSS in the data set?

scorecard_fos_tn[scorecard_fos_tn["inst_name"].str.contains("Nashville Software", case=False, na=False)]

,unitid,inst_name,cipcode,median_earnings_1yr,median_earnings_3yr,median_earnings_5yr,cip4


In [34]:
scorecard_fos_tn["cipcode"].value_counts()

cipcode
5202    134
5138    113
2401     85
5109     73
1312     68
       ... 
405       1
406       1
105       1
1106      1
5126      1
Name: count, Length: 292, dtype: int64

In [35]:
scorecard_fos_tn["cipcode"].unique()

array([2401, 3802, 3906, 3999, 4400, 5009, 5207, 1204, 1504, 4603, 4701,
       4703, 4706, 4805, 5108, 5139, 5204,  100,  901,  904, 1101, 1105,
       1107, 1108, 1301, 1303, 1304, 1306, 1310, 1311, 1312, 1313, 1314,
       1412, 1500, 1506, 1601, 1612, 2301, 2601, 2611, 2701, 3105, 3801,
       4005, 4006, 4008, 4201, 4228, 4301, 4407, 4510, 4511, 4901, 5005,
       5007, 5107, 5109, 5110, 5111, 5115, 5138, 5201, 5202, 5203, 5208,
       5214, 5401, 5122,  301,  402,  408,  501,  909,  910, 1002, 1399,
       1605, 1609, 1611, 2200, 2201, 2602, 2615, 2703, 3099, 3800, 3902,
       3905, 4506, 4509, 5004, 5006, 5010, 5120, 5123, 5206, 5211, 5212,
       5213, 5219, 2613, 3005, 4299, 4303, 5100, 1401, 2313, 3001, 3008,
       3010, 3071, 3904, 5299, 1499, 1901, 1902, 1904, 1905, 1906, 1907,
       1909, 2699, 3101, 3899, 4402, 4599, 5131, 5132,  106,  183, 1503,
       1507, 1508, 1513, 2203, 3000, 3201, 4103, 4302, 4702, 4902, 5106,
       5135, 5199, 5209,  502, 1110, 1407, 1408, 14

In [38]:
completions["cip4"].value_counts()

cip4
9900    41384
5202    18118
1313    17387
1312    12399
5109    10368
        ...  
1206        1
3035        1
3043        1
3048        1
2906        1
Name: count, Length: 406, dtype: int64

In [39]:
completions["cip4"].unique()

array(['0909', '1001', '1101', '1104', '1304', '1399', '1503', '1504',
       '1505', '1506', '1507', '1510', '2203', '2609', '2902', '2904',
       '2999', '3106', '3199', '4004', '4199', '4301', '4302', '4303',
       '4407', '4706', '4901', '4999', '5006', '5009', '5106', '5107',
       '5108', '5109', '5110', '5115', '5118', '5122', '5131', '5199',
       '5202', '5203', '5210', '5401', '9900', '0100', '0101', '0109',
       '0110', '0111', '0199', '0305', '0403', '1002', '1310', '1312',
       '1313', '1408', '1410', '1419', '1499', '1502', '1508', '1901',
       '2301', '2401', '2601', '2701', '4005', '4008', '4201', '4228',
       '4506', '4510', '4511', '5007', '5102', '5208', '5214', '0502',
       '0901', '1301', '1311', '1401', '1405', '1409', '1414', '1418',
       '1601', '2602', '2604', '2605', '2608', '2610', '2611', '2613',
       '2615', '2703', '3019', '3099', '3105', '3801', '4404', '4502',
       '4599', '5001', '5005', '5104', '5105', '5112', '5117', '5123',
      

In [42]:
completions.dtypes

unitid                int64
cipcode              object
credlev               int64
completions           int64
completions_men       int64
completions_women     int64
year                  int64
credential_group     object
cip4                 object
dtype: object

In [43]:
scorecard_fos_tn.dtypes

unitid                 float64
inst_name               object
cipcode                  int64
median_earnings_1yr     object
median_earnings_3yr     object
median_earnings_5yr     object
cip4                    object
dtype: object

In [45]:
scorecard_il_tn.dtypes

unitid                  int64
state                  object
tuition_in            float64
tuition_out           float64
avg_net_price_pub     float64
avg_net_price_priv    float64
avg_annual_cost       float64
dtype: object

In [48]:
scorecard_il_tn.shape

(153, 7)

In [50]:
scorecard_il_tn.head()


,unitid,state,tuition_in,tuition_out,avg_net_price_pub,avg_net_price_priv,avg_annual_cost
2961,219505,TN,12474.0,12474.0,NaN,19294.0,27241.0
2962,219587,TN,NaN,NaN,NaN,10343.0,NaN
2963,219596,TN,NaN,NaN,3720.0,NaN,NaN
2964,219602,TN,8675.0,14219.0,14846.0,NaN,26790.0
2965,219639,TN,13846.0,13846.0,NaN,13401.0,21438.0


In [51]:
scorecard_fos_tn.head()

,unitid,inst_name,cipcode,median_earnings_1yr,median_earnings_3yr,median_earnings_5yr,cip4
162267,219505.0,American Baptist College,2401,PS,PS,PS,2401
162268,219505.0,American Baptist College,2401,PS,PS,PS,2401
162269,219505.0,American Baptist College,2401,PS,PS,PS,2401
162270,219505.0,American Baptist College,3802,PS,PS,PS,3802
162271,219505.0,American Baptist College,3906,PS,PS,PS,3906
